In [1]:

# ============================================================
# MODULE 4.1 — PDF LOGICAL SPLITTING & CLASSIFICATION
# ============================================================

# %%
# ── CELL 1: CÀI ĐẶT ─────────────────────────────────────────
!pip install pymupdf pdfplumber pdf2image pytesseract -q
!apt-get install -y tesseract-ocr tesseract-ocr-vie -q

import fitz, pdfplumber, re, json, uuid, time
import pytesseract
from pdf2image import convert_from_path
from datetime import datetime
from pathlib import Path
from google.colab import drive
print("✅ Setup xong!")

# %%
# ── CELL 2: CONFIG & GROUND TRUTH ĐÃ CHUẨN HÓA ──────────────
PDF_PATH = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/mix_doc/document_merged2.pdf'

RE_SO_VAN_BAN = r'\d{1,3}/\d{4}/[A-ZĐQĐ]+[\w\-]*'

GT_BOUNDARIES = {1, 4, 7, 9, 29, 35, 43, 44, 50, 52, 54, 57, 59, 62, 63, 66, 75, 81, 103, 105, 108}
GT_SEPARATORS = {65}

# Đã fix lại Ground Truth khớp chính xác với bản chất của các số văn bản trong Header
GT_SEGMENT_TYPES = {
    1: "Quyết định",           # 200/2026/QĐ-PT
    4: "Quyết định",           # 14/2026/QĐ-PT
    7: "Quyết định",           # 81/2026/QĐ-PT
    9: "Bản án dân sự",        # 453/2026/DS-PT
    29: "Quyết định",          # Quyết định đình chỉ
    35: "Bản án dân sự",       # HNGĐ
    43: "Bản án hình sự",
    44: "Bản án hình sự",      # 05/2016/HSST
    50: "Quyết định",          # 36/2026/QĐ-PT
    52: "Quyết định",          # 07/2026/QĐ-PT
    54: "Quyết định",          # 26/2026/QĐ-PT
    57: "Quyết định",          # 11/2026/QĐ-PT
    59: "Quyết định",          # 21/2026/QĐ-PT
    62: "Quyết định",          # 01/2025/QĐST-HC
    63: "Quyết định",          # 01/2026/QĐ-PT
    66: "Bản án hình sự",      # 04/2017/HSST
    75: "Bản án dân sự",       # 03/2017/DS-ST
    81: "Bản án hành chính",   # 427/2019/HC-PT
    103: "Quyết định",         # 46/2025/QĐST-HC
    105: "Quyết định",         # 140/2025/QĐST-HC
    108: "Quyết định",         # 913/2025/QĐÐST-HC
}

try:
    with fitz.open(PDF_PATH) as doc:
        TOTAL_PAGES = len(doc)
    print(f"✅ Config xong | {PDF_PATH.split('/')[-1]} | {TOTAL_PAGES} trang")
except Exception as e:
    print(f"⚠️ Vui lòng kiểm tra lại PDF_PATH: {e}")
    TOTAL_PAGES = 109 # Fallback for structural building

# %%
# ── CELL 3: EXTRACT TEXT & FEATURES ────────────────────────
from PIL import Image

def get_text_hybrid_fast(pdf_path: str, page_idx: int) -> str:
    with fitz.open(pdf_path) as doc:
        page = doc[page_idx]
        text = page.get_text("text").strip()
        # Chuyển sang OCR nếu là PDF scan (ít text gốc)
        if len(text) < 10:
            pix = page.get_pixmap(dpi=150, colorspace=fitz.csGRAY)
            img = Image.frombytes("L", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, lang='vie').strip()
    return text

def extract_features(pdf_path: str, page_idx: int) -> dict:
    text = get_text_hybrid_fast(pdf_path, page_idx)
    lines = text.split("\n") if text else []
    n = len(lines)

    header = " ".join(l.strip() for l in lines[:max(5, int(n * 0.20))] if l.strip())
    header_up = header.upper()

    has_quoc_hieu = "CỘNG HÒA XÃ HỘI" in header_up
    has_tieu_ngu = "HẠNH PHÚC" in header_up
    has_toa_an = "TÒA ÁN NHÂN DÂN" in header_up
    so_vb_header = re.findall(RE_SO_VAN_BAN, header)

    return {
        "page_num": page_idx + 1,
        "char_count": len(text),
        "is_blank": len(text) < 30,
        "header": header,
        "has_quoc_hieu": has_quoc_hieu,
        "has_tieu_ngu": has_tieu_ngu,
        "has_toa_an": has_toa_an,
        "so_vb_header": so_vb_header,
        "text_full": text,
    }

# Để tiết kiệm thời gian chạy mẫu, bạn có thể uncomment phần gọi hàm khi build thật.
print("⏳ Đã định nghĩa logic Hybrid Extraction.")

# %%
# ── CELL 4: BOUNDARY DETECTION ─────────────────────────────
def extract_strict_header(text: str, n_lines: int = 5) -> tuple[str, str]:
    lines = text.split("\n") if text else []
    result = []
    first_line = ""
    for line in lines:
        stripped = line.strip()
        if not stripped or re.match(r'^\d{1,3}$', stripped): continue
        if not first_line: first_line = stripped
        result.append(stripped)
        if len(result) >= n_lines: break
    return " ".join(result), first_line

FIRST_LINE_PATTERNS = [
    r'TÒA ÁN NHÂN DÂN', r'VIỆN KIỂM SÁT', r'CỘNG HÒA XÃ HỘI',
    r'NHÂN DANH', r'QUYẾT ĐỊNH\s*$'
]

def is_doc_start_header(strict_header: str, first_line: str) -> tuple[bool, str]:
    h = strict_header.upper()
    first_up = first_line.upper()

    if not any(re.search(p, first_up) for p in FIRST_LINE_PATTERNS):
        return False, ""

    group_a = ("CỘNG HÒA XÃ HỘI" in h or "ĐỘC LẬP" in h or "HẠNH PHÚC" in h)
    group_b = ("TÒA ÁN NHÂN DÂN" in h or "VIỆN KIỂM SÁT" in h)
    group_c = (bool(re.search(RE_SO_VAN_BAN, strict_header)) or "BẢN ÁN SỐ" in h or "NHÂN DANH" in h or bool(re.search(r'QUYẾT ĐỊNH\s*$', h, re.MULTILINE)))

    groups_met = sum([group_a, group_b, group_c])
    if groups_met >= 2:
        signals = []
        if group_a: signals.append("quoc_hieu")
        if group_b: signals.append("co_quan")
        if group_c: signals.append("dinh_danh")
        return True, "+".join(signals)
    return False, ""

def is_boundary(feat: dict, prev: dict | None) -> tuple[bool, str, float]:
    if prev is None: return True, "first_page", 1.0
    if feat["is_blank"]: return False, "blank_page", 0.0
    if prev["is_blank"]: return True, "after_blank", 0.95

    strict_hdr, first_line = feat["strict_header"]
    is_start, signal = is_doc_start_header(strict_hdr, first_line)
    if is_start:
        return True, signal, 1.0 if "quoc_hieu" in signal and "co_quan" in signal else 0.9
    return False, "continuation", 0.0

# %%
# ── CELL 5: STAGE 3 — SEGMENT CLASSIFICATION (FIXED) ────────
# Tối ưu nhận diện theo Ký hiệu chuẩn của Tòa Án
def classify_segment(pages: list) -> tuple[str, float, dict]:
    first_page_text = pages[0]["text_full"].upper()

    # Ưu tiên 1: Quét mã văn bản ở Trang 1 và Trang 2
    so_vb_matches = re.findall(RE_SO_VAN_BAN, first_page_text)
    if not so_vb_matches and len(pages) > 1:
        so_vb_matches = re.findall(RE_SO_VAN_BAN, pages[1]["text_full"].upper())

    label = None
    scores = {"Bản án hành chính": 0, "Bản án hình sự": 0, "Bản án dân sự": 0, "Quyết định": 0}

    # Bóc tách qua số hiệu
    if so_vb_matches:
        so_vb = so_vb_matches[0]
        if 'QĐ' in so_vb: label = "Quyết định"
        elif 'HC' in so_vb: label = "Bản án hành chính"
        elif 'HS' in so_vb: label = "Bản án hình sự"
        elif any(x in so_vb for x in ['DS', 'HNGĐ', 'KDTM']): label = "Bản án dân sự"

    # Ưu tiên 2: Semantic Keywords ở đoạn đầu văn bản
    if not label:
        header_context = first_page_text[:1000]
        if "QUYẾT ĐỊNH" in header_context: label = "Quyết định"
        elif "HÀNH CHÍNH" in header_context: label = "Bản án hành chính"
        elif "HÌNH SỰ" in header_context: label = "Bản án hình sự"
        elif any(k in header_context for k in ["DÂN SỰ", "HÔN NHÂN", "KINH DOANH"]): label = "Bản án dân sự"

    # Fallback
    if not label: label = "Quyết định"

    # Tính toán độ tin cậy
    conf = 0.98 if so_vb_matches else 0.85
    scores[label] = 10.0 # Mockup score
    return label, conf, scores

def group_segments(all_features: list, boundary_pages: set, separator_pages: set) -> list:
    segments, current = [], []
    for f in all_features:
        pn = f["page_num"]
        if f["is_blank"] or pn in separator_pages:
            if current:
                segments.append(current)
                current = []
            continue
        if pn in boundary_pages and current:
            segments.append(current)
            current = []
        current.append(f)
    if current: segments.append(current)
    return segments

# %%
# ── CELL 6: STAGE 4 — FULL PIPELINE & CHUẨN HÓA API JSON ──────
OUTPUT_DIR = '/content/drive/MyDrive/VNDigitizeComprehensiveSystem_Team03/dataset_module4/output'

def full_pipeline(pdf_path: str, output_dir: str, gt_boundaries: set = None, gt_segment_types: dict = None) -> dict:
    t_total_start = time.time()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # 1. Extract
    features = []
    try:
        with fitz.open(pdf_path) as doc: n_pages = len(doc)
        for i in range(n_pages):
            f = extract_features(pdf_path, i)
            f["strict_header"] = extract_strict_header(f["text_full"], n_lines=5)
            features.append(f)
    except:
        return {"error": "Không thể xử lý PDF."}

    # 2. Boundary
    pred_boundaries, pred_separators = set(), set()
    for i, feat in enumerate(features):
        prev = features[i-1] if i > 0 else None
        bdry, _, _ = is_boundary(feat, prev)
        if bdry: pred_boundaries.add(feat["page_num"])
        if feat["is_blank"]: pred_separators.add(feat["page_num"])

    # 3. Classify
    segs = group_segments(features, pred_boundaries, pred_separators)
    cls_results = []
    for i, pages in enumerate(segs):
        page_start = pages[0]["page_num"]
        page_end = pages[-1]["page_num"]
        label, conf, scores = classify_segment(pages)

        cls_results.append({
            "segment": i + 1,
            "page_start": page_start,
            "page_end": page_end,
            "n_pages": len(pages),
            "type": label,
            "confidence": conf
        })

    # 4. Split & Construct Output (Chuẩn hóa theo JSON Schema từ Requirement [cite: 49, 50, 51, 63, 64])
    src = fitz.open(pdf_path)
    doc_id = str(uuid.uuid4())[:8]
    sub_docs = []

    for r in cls_results:
        label_safe = r["type"].replace(" ", "_")
        filename = f"sub_{r['segment']:03d}_{label_safe}_p{r['page_start']}-{r['page_end']}.pdf"
        out_path = f"{output_dir}/{filename}"

        doc_out = fitz.open()
        doc_out.insert_pdf(src, from_page=r["page_start"] - 1, to_page=r["page_end"] - 1)
        doc_out.save(out_path)
        doc_out.close()

        # Ánh xạ chuẩn theo API Contract [cite: 63, 64, 65]
        sub_docs.append({
            "type": r["type"],
            "page_start": r["page_start"],
            "page_end": r["page_end"]
        })
    src.close()

    t_total_ms = int((time.time() - t_total_start) * 1000)

    # Payload chuẩn theo tài liệu yêu cầu (Mục 5) [cite: 49-67]
    result_json = {
        "document_id": doc_id,
        "classification": "Tài liệu hỗn hợp (Merged Document)",
        "summary": "Tập tin PDF chứa nhiều tài liệu đã được AI tự động phân loại và tách lớp.",
        "confidence_overall": round(sum(r["confidence"] for r in cls_results) / len(cls_results), 2) if cls_results else 0.0,
        "metadata": [],
        "sub_documents": sub_docs
    }

    json_path = f"{output_dir}/result_{doc_id}.json"
    with open(json_path, "w", encoding="utf-8") as fj:
        json.dump(result_json, fj, ensure_ascii=False, indent=2)

    return result_json


print("⏳ Đang xử lý Pipeline...")
result = full_pipeline(PDF_PATH, OUTPUT_DIR, GT_BOUNDARIES, GT_SEGMENT_TYPES)
print(json.dumps(result, ensure_ascii=False, indent=2))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 104.5 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-vie
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 417 kB of archives.
After this operation, 546 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-vie all 1:4.00~git30-7274cfa-1.1 [417 kB]
Fetched 417 kB in 0s (3,123 kB/s)
Sel